In [1]:
import pandas as pd
from database import get_connection
from pathlib import Path

conn = get_connection()

def query_df(conn, sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=columns)

In [2]:
posts_df = query_df(conn, "SELECT * FROM posts")
comments_df = query_df(conn, "SELECT * FROM comments")

posts_df.info()
print("\n")
comments_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2415 entries, 0 to 2414
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype                           
---  ------             --------------  -----                           
 0   post_id            2415 non-null   str                             
 1   created_utc        2415 non-null   datetime64[us, America/New_York]
 2   title              2415 non-null   str                             
 3   is_self            2415 non-null   bool                            
 4   score              2415 non-null   int64                           
 5   ups                2415 non-null   int64                           
 6   upvote_ratio       2415 non-null   float64                         
 7   author             2415 non-null   str                             
 8   author_flair_text  819 non-null    str                             
 9   location_lat       0 non-null      object                          
 10  location_long      0 no

In [3]:
# Post cleaning objects
remove_post_columns = ['location_lat', 'location_long', 'location_name', 'category', 'num_reports']
deleted_or_removed = ['[deleted]', '[removed]']

clean_posts_df = posts_df.drop(columns=remove_post_columns) # removing empty columns
clean_posts_df['selftext'] = clean_posts_df['selftext'].replace(deleted_or_removed, "") # preserving documnents with titles but no body text
clean_posts_df['full_text'] = clean_posts_df['title'] + " " + clean_posts_df['selftext'] # combining title and selftext as one field

# Comment cleaning objects
remove_comment_columns = ['downs']
moderator_authors = ['AutoModerator', 'EntireAd9022', 'OTTO-M0D', 'Ti_xiv', 'W0NT0NY', '[deleted]', 'LoveAndDeepspace-ModTeam']

clean_comments_df = comments_df.drop(columns=remove_comment_columns)
clean_comments_df = clean_comments_df[clean_comments_df['author'].isin(moderator_authors) == False] # Remove moderator comments
clean_comments_df = clean_comments_df[~clean_comments_df['body'].isin(deleted_or_removed)] # Remove deleted and removed comments

In [4]:
output_dir = Path('../analysis/data')
output_dir.mkdir(parents=True, exist_ok=True)

clean_posts_df.to_parquet(output_dir / 'clean_posts.parquet')
clean_comments_df.to_parquet(output_dir / 'clean_comments.parquet')
conn.close()